In [ ]:
%gui qt
%load_ext autoreload
%autoreload 2

In [ ]:
import hmt_v3 as hmt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
me3_raw_df = pd.read_csv("test_data/k27_k27_thaw009_me3.csv")
ac_raw_df = pd.read_csv("test_data/k27_k27_thaw009_ac.csv")

me3_filtered_df = hmt.preprocess.filter_axial(me3_raw_df)
ac_filtered_df = hmt.preprocess.filter_axial(ac_raw_df)

binary_mask, me3_df, ac_df = hmt.preprocess.binarize_nucleus(me3_filtered_df, ac_filtered_df, thresh=2, bin_size=50, show_plots=True)
distance_map, contour_bands = hmt.preprocess.create_radial_contours(binary_mask, show_plots=True)


In [ ]:
step=10
print("Extracting emperical H3K27me3 distributions...")
me3_rdf, me3_adf = hmt.simulate.extract_empirical_parameters(me3_df, sdis=500, step=step)

print("Extracting emperical H3K27ac distributions...")
ac_rdf, ac_adf = hmt.simulate.extract_empirical_parameters(ac_df, sdis=500, step=step)

hmt.visualize.plot_rdf_adf(me3_rdf, me3_adf, ac_rdf, ac_adf, step=10)

In [ ]:
# ── RDF/ADF: 20 innermost vs 20 outermost vs entire nucleus ────────────────
def band_of(df, contour_bands, x_min, y_min, bin_size=50):
    coords = df[["x [nm]", "y [nm]"]].to_numpy()
    x_idx = np.clip(((coords[:, 0] - x_min) / bin_size).astype(int), 0, contour_bands.shape[0] - 1)
    y_idx = np.clip(((coords[:, 1] - y_min) / bin_size).astype(int), 0, contour_bands.shape[1] - 1)
    return contour_bands[x_idx, y_idx]

x_min, y_min = hmt.preprocess.mask_origin(me3_filtered_df, ac_filtered_df)
me3_band = band_of(me3_df, contour_bands, x_min, y_min)
ac_band  = band_of(ac_df,  contour_bands, x_min, y_min)

step = 10
regions = {
    "4 Innermost Contours":   (0, 19),
    "Entire Nucleus":         (0, 99),
    "4 Outermost Contours":   (80, 99),
}

for label, (lo, hi) in regions.items():
    me3_sub = me3_df[(me3_band >= lo) & (me3_band <= hi)]
    ac_sub  = ac_df[(ac_band  >= lo) & (ac_band  <= hi)]

    me3_rdf_sub, me3_adf_sub = hmt.simulate.extract_empirical_parameters(me3_sub, sdis=500, step=step)
    ac_rdf_sub,  ac_adf_sub  = hmt.simulate.extract_empirical_parameters(ac_sub,  sdis=500, step=step)

    print(f"--- {label} ---  me3: {len(me3_sub):,} locs | ac: {len(ac_sub):,} locs")
    hmt.visualize.plot_rdf_adf(
        me3_rdf_sub, me3_adf_sub, ac_rdf_sub, ac_adf_sub, step=step,
        rdf_title=f"RDF: {label}",
        adf_title=f"ADF: {label}",
    )


In [ ]:
noise_fraction = 0.05  # fraction of domain localizations to add as uniform background noise

for spacing in [600, 1000, 1500]:
    grid_seeds = hmt.simulate.make_grid_seeds(
        n_rows=10, n_cols=10,
        spacing=spacing,
        z_values=me3_df["z [nm]"].values,
    )

    n_locs = hmt.simulate.extract_n_locs_from_rdf(me3_rdf, step=step)

    grid_sim = hmt.simulate.spawn_nanodomains(
        grid_seeds,
        rdf=me3_rdf, adf=me3_adf,
        n_locs=n_locs,
        step=step,
    )

    grid_noise = hmt.simulate.add_sim_noise(grid_sim, noise_fraction=noise_fraction)
    grid_final = pd.concat([grid_sim, grid_noise], ignore_index=True)

    print(f"Grid: {len(grid_seeds)} seeds  |  {len(grid_sim):,} domain locs  |  {len(grid_noise):,} noise locs  |  spacing = {spacing:.0f} nm")

    fig, axes = plt.subplots(1, 2, figsize=(9.5, 5))
    hmt.visualize.plot_nanodomain_2d(grid_sim,   grid_seeds, title="No noise",                          ax=axes[0])
    hmt.visualize.plot_nanodomain_2d(grid_final, grid_seeds, title=f"With noise ({noise_fraction:.0%})", ax=axes[1])
    fig.suptitle(f"Spacing = {spacing:.0f} nm")
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Sensitivity Analysis: grid spacing × noise fraction × use_z ───────────────
spacings        = [500, 750, 1000, 1500]   # nm between seed centres
noise_fractions = [0.0, 0.1, 0.3, 0.5]    # fraction of domain locs added as uniform noise
use_z_options   = [False, True]            # 2-D (XY) vs 3-D (XYZ) clustering

n_rows, n_cols = 10, 10
eps_grid       = np.arange(10, 225, 5.0)  # nm — coarse sweep over candidate epsilons
min_samples    = 8

n_locs = hmt.simulate.extract_n_locs_from_rdf(me3_rdf, step=step)

records = []
print("Running sensitivity analysis...")
print(f"  {len(spacings)} spacings  ×  {len(noise_fractions)} noise fractions  "
      f"×  {len(use_z_options)} z-modes  =  {len(spacings)*len(noise_fractions)*len(use_z_options)} conditions\n")

for spacing in spacings:
    # Build one ground-truth simulation per spacing; reuse it across noise/z sweeps
    seeds  = hmt.simulate.make_grid_seeds(
        n_rows=n_rows, n_cols=n_cols,
        spacing=spacing,
        z_values=me3_df["z [nm]"].values,
    )
    sim_df = hmt.simulate.spawn_nanodomains(
        seeds, rdf=me3_rdf, adf=me3_adf, n_locs=n_locs, step=step,
    )

    for noise_frac in noise_fractions:
        noise_df = hmt.simulate.add_sim_noise(sim_df, noise_fraction=noise_frac)
        full_df  = pd.concat([sim_df, noise_df], ignore_index=True)

        coords      = full_df[["x [nm]", "y [nm]", "z [nm]"]].to_numpy()
        true_labels = full_df["cluster_label"].to_numpy()

        for use_z in use_z_options:
            best_eps  = eps_grid[0]
            best_cost = 2.0  # worst possible 1-ARI

            for test_eps in eps_grid:
                cost = hmt.simulate.epsilon_cost_cluster(
                    test_eps, coords, true_labels, min_samples, use_z
                )
                if cost < best_cost:
                    best_cost = cost
                    best_eps  = test_eps

            best_ari = 1.0 - best_cost
            records.append({
                "spacing_nm":     spacing,
                "noise_fraction": noise_frac,
                "use_z":          use_z,
                "best_eps_nm":    best_eps,
                "best_ARI":       best_ari,
            })
            print(f"  spacing={spacing:5d} nm  noise={noise_frac:.0%}  use_z={str(use_z):5}  "
                  f"→  eps={best_eps:6.1f} nm  ARI={best_ari:.4f}")

results_df = pd.DataFrame(records)
print("\nDone.")

In [ ]:
# ── Heatmaps: ARI and optimal epsilon ─────────────────────────────────────────
metrics     = ["best_ARI",  "best_eps_nm"]
metric_labs = ["Best ARI (higher = better)", "Optimal Epsilon (nm)"]
cmaps       = ["RdYlGn",    "viridis"]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Sensitivity Analysis — Heatmaps", fontsize=14)

for row, (metric, label, cmap) in enumerate(zip(metrics, metric_labs, cmaps)):
    vmin = results_df[metric].min()
    vmax = results_df[metric].max()

    for col, use_z in enumerate([False, True]):
        ax     = axes[row, col]
        subset = results_df[results_df["use_z"] == use_z]
        pivot  = subset.pivot(index="spacing_nm", columns="noise_fraction", values=metric)

        im = ax.imshow(pivot.values, aspect="auto", origin="lower",
                       cmap=cmap, vmin=vmin, vmax=vmax)

        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([f"{v:.0%}" for v in pivot.columns])
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels([f"{int(v)}" for v in pivot.index])
        ax.set_xlabel("Noise Fraction")
        ax.set_ylabel("Grid Spacing (nm)")
        ax.set_title(f"{'3D (XYZ)' if use_z else '2D (XY)'}  —  {label}")

        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                val = pivot.values[i, j]
                txt = f"{val:.3f}" if metric == "best_ARI" else f"{val:.0f}"
                ax.text(j, i, txt, ha="center", va="center", fontsize=9)

        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
# ── Line plots: ARI vs noise fraction, one line per spacing, panels for z ─────
markers = ["o", "s", "^", "D"]
colors  = plt.cm.tab10(np.linspace(0, 0.6, len(spacings)))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("ARI vs Noise Fraction by Grid Spacing", fontsize=13)

for col, use_z in enumerate([False, True]):
    ax     = axes[col]
    subset = results_df[results_df["use_z"] == use_z]

    for j, spacing in enumerate(spacings):
        grp = subset[subset["spacing_nm"] == spacing].sort_values("noise_fraction")
        ax.plot(grp["noise_fraction"], grp["best_ARI"],
                marker=markers[j % len(markers)], color=colors[j],
                linewidth=1.8, label=f"{spacing} nm")

    ax.set_xlabel("Noise Fraction")
    ax.set_ylabel("Best ARI")
    ax.set_title(f"{'3D (XYZ)' if use_z else '2D (XY)'}")
    ax.set_xlim(-0.02, max(noise_fractions) + 0.02)
    ax.set_ylim(0, 1.05)
    ax.legend(title="Grid Spacing")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Direct 2D vs 3D comparison at each condition ──────────────────────────────
fig, axes = plt.subplots(1, len(noise_fractions), figsize=(14, 4), sharey=True)
fig.suptitle("2D vs 3D ARI by Noise Fraction", fontsize=13)

for ax, noise_frac in zip(axes, noise_fractions):
    sub2d = results_df[(results_df["use_z"] == False) & (results_df["noise_fraction"] == noise_frac)]
    sub3d = results_df[(results_df["use_z"] == True)  & (results_df["noise_fraction"] == noise_frac)]

    x = np.arange(len(spacings))
    w = 0.35
    ax.bar(x - w/2, sub2d.sort_values("spacing_nm")["best_ARI"], width=w, label="2D (XY)",  color="steelblue")
    ax.bar(x + w/2, sub3d.sort_values("spacing_nm")["best_ARI"], width=w, label="3D (XYZ)", color="tomato")

    ax.set_xticks(x)
    ax.set_xticklabels([f"{s}" for s in spacings], rotation=45)
    ax.set_xlabel("Grid Spacing (nm)")
    ax.set_title(f"Noise = {noise_frac:.0%}")
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.3)
    if ax is axes[0]:
        ax.set_ylabel("Best ARI")
        ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Density subsampling sweep ──────────────────────────────────────────────────
# For each spacing, generate a full simulation (100 seeds + reference noise),
# then subsample ALL localizations to simulate lower labeling efficiency.
# Sweep from 100 % down to 50 % in 10 % steps.

density_fractions = np.linspace(1.0, 0.2, 9)  # 1.0, 0.9, 0.8, 0.7, 0.6, 0.5
reference_noise   = 0.1                         # fixed noise level for this sweep

density_records = []
print("Running density subsampling sweep...")
print(f"  {len(spacings)} spacings  ×  {len(density_fractions)} density fractions  "
      f"×  {len(use_z_options)} z-modes  =  "
      f"{len(spacings)*len(density_fractions)*len(use_z_options)} conditions\n")

rng = np.random.default_rng(42)  # fixed seed so subsampling is reproducible

for spacing in spacings:
    seeds  = hmt.simulate.make_grid_seeds(
        n_rows=n_rows, n_cols=n_cols,
        spacing=spacing,
        z_values=me3_df["z [nm]"].values,
    )
    sim_df   = hmt.simulate.spawn_nanodomains(
        seeds, rdf=me3_rdf, adf=me3_adf, n_locs=n_locs, step=step,
    )
    noise_df = hmt.simulate.add_sim_noise(sim_df, noise_fraction=reference_noise)
    full_df  = pd.concat([sim_df, noise_df], ignore_index=True)
    full_idx = np.arange(len(full_df))

    for density_frac in density_fractions:
        n_keep  = max(min_samples + 1, int(len(full_df) * density_frac))
        chosen  = rng.choice(full_idx, size=n_keep, replace=False)
        sub_df  = full_df.iloc[chosen].reset_index(drop=True)

        coords      = sub_df[["x [nm]", "y [nm]", "z [nm]"]].to_numpy()
        true_labels = sub_df["cluster_label"].to_numpy()

        for use_z in use_z_options:
            best_eps  = eps_grid[0]
            best_cost = 2.0

            for test_eps in eps_grid:
                cost = hmt.simulate.epsilon_cost_cluster(
                    test_eps, coords, true_labels, min_samples, use_z
                )
                if cost < best_cost:
                    best_cost = cost
                    best_eps  = test_eps

            best_ari = 1.0 - best_cost
            density_records.append({
                "spacing_nm":       spacing,
                "density_fraction": round(density_frac, 2),
                "use_z":            use_z,
                "best_eps_nm":      best_eps,
                "best_ARI":         best_ari,
                "n_locs":           n_keep,
            })
            print(f"  spacing={spacing:5d} nm  density={density_frac:.0%}  use_z={str(use_z):5}  "
                  f"→  eps={best_eps:6.1f} nm  ARI={best_ari:.4f}  ({n_keep:,} locs)")

density_df = pd.DataFrame(density_records)
print("\nDone.")

In [ ]:
# ── Density sweep plots ────────────────────────────────────────────────────────
# Labels ordered high → low density (100 % … 20 %)
pct_labels = [f"{v:.0%}" for v in sorted(density_df["density_fraction"].unique(), reverse=True)]

# -- 1. Line plot: ARI vs density fraction, high → low on x-axis
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
fig.suptitle(f"ARI vs Labeling Density  (noise = {reference_noise:.0%})", fontsize=13)

for col, use_z in enumerate([False, True]):
    ax     = axes[col]
    subset = density_df[density_df["use_z"] == use_z]

    for j, spacing in enumerate(spacings):
        grp = subset[subset["spacing_nm"] == spacing].sort_values("density_fraction", ascending=False)
        ax.plot(grp["density_fraction"], grp["best_ARI"],
                marker=markers[j % len(markers)], color=colors[j],
                linewidth=1.8, label=f"{spacing} nm")

    ax.set_xlabel("Density Fraction (subsampled)")
    ax.set_ylabel("Best ARI")
    ax.set_title(f"{'3D (XYZ)' if use_z else '2D (XY)'}")
    ax.set_xticks(sorted(density_df["density_fraction"].unique(), reverse=True))
    ax.set_xticklabels(pct_labels)
    ax.set_xlim(1.05, min(density_df["density_fraction"]) - 0.05)  # high → low
    ax.set_ylim(0, 1.05)
    ax.legend(title="Grid Spacing")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# -- 2. Heatmaps: columns ordered high → low density
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle(f"Density Subsampling — Heatmaps  (noise = {reference_noise:.0%})", fontsize=13)

for row, (metric, label, cmap) in enumerate(zip(
        ["best_ARI",  "best_eps_nm"],
        ["Best ARI (higher = better)", "Optimal Epsilon (nm)"],
        ["RdYlGn",    "viridis"])):

    vmin = density_df[metric].min()
    vmax = density_df[metric].max()

    for col, use_z in enumerate([False, True]):
        ax     = axes[row, col]
        subset = density_df[density_df["use_z"] == use_z]
        pivot  = subset.pivot(index="spacing_nm", columns="density_fraction", values=metric)
        pivot  = pivot[pivot.columns[::-1]]  # reverse: high density on the left

        im = ax.imshow(pivot.values, aspect="auto", origin="lower",
                       cmap=cmap, vmin=vmin, vmax=vmax)

        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([f"{v:.0%}" for v in pivot.columns])
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels([f"{int(v)}" for v in pivot.index])
        ax.set_xlabel("Density Fraction")
        ax.set_ylabel("Grid Spacing (nm)")
        ax.set_title(f"{'3D (XYZ)' if use_z else '2D (XY)'}  —  {label}")

        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                val = pivot.values[i, j]
                txt = f"{val:.3f}" if metric == "best_ARI" else f"{val:.0f}"
                ax.text(j, i, txt, ha="center", va="center", fontsize=9)

        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# -- 3. ΔARI bars ordered high → low density
fig, axes = plt.subplots(1, len(spacings), figsize=(14, 4), sharey=True)
fig.suptitle("ΔARI (3D − 2D) by Density Fraction", fontsize=13)

for ax, spacing in zip(axes, spacings):
    sub2d = density_df[(density_df["use_z"] == False) & (density_df["spacing_nm"] == spacing)].sort_values("density_fraction", ascending=False)
    sub3d = density_df[(density_df["use_z"] == True)  & (density_df["spacing_nm"] == spacing)].sort_values("density_fraction", ascending=False)
    delta = sub3d["best_ARI"].values - sub2d["best_ARI"].values

    bar_colors = ["tomato" if d < 0 else "steelblue" for d in delta]
    ax.bar(range(len(delta)), delta, color=bar_colors)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xticks(range(len(pct_labels)))
    ax.set_xticklabels(pct_labels, rotation=45)
    ax.set_xlabel("Density Fraction")
    ax.set_title(f"Spacing = {spacing} nm")
    ax.grid(axis="y", alpha=0.3)
    if ax is axes[0]:
        ax.set_ylabel("ΔARI  (3D − 2D)")

plt.tight_layout()
plt.show()

In [ ]:
x_min = pd.concat([me3_df, ac_df])["x [nm]"].min()
y_min = pd.concat([me3_df, ac_df])["y [nm]"].min()

prof = hmt.simulate.extract_radial_density_profile(me3_df, contour_bands, x_min, y_min)
print("bands with zero me3 density:", np.where(prof == 0)[0])          # expect a run of low (central) bands
print("innermost 15 bands (center→out):", np.round(prof[:15], 5))       # expect ~0 in the center


In [ ]:
# (1) Origin check — the grid was built from the PRE-mask inputs
true_x0 = pd.concat([me3_filtered_df, ac_filtered_df])["x [nm]"].min()
true_y0 = pd.concat([me3_filtered_df, ac_filtered_df])["y [nm]"].min()
print("mask origin (pre-mask):", true_x0, true_y0)
print("x_min / y_min passed  :", x_min, y_min)
print("offset (nm)           :", x_min - true_x0, y_min - true_y0)   # want ~0

# (2) Visual — me3 (red) and ac (blue) over the bands, mapped with the TRUE origin
xm = ((me3_df['x [nm]'] - true_x0)/50).astype(int); ym = ((me3_df['y [nm]'] - true_y0)/50).astype(int)
xa = ((ac_df['x [nm]']  - true_x0)/50).astype(int); ya = ((ac_df['y [nm]']  - true_y0)/50).astype(int)
fig, ax = plt.subplots(figsize=(7,7))
ax.imshow(np.ma.masked_less(contour_bands, 0).T, origin='lower', cmap='Greys', alpha=0.6)
ax.scatter(xm, ym, s=0.3, c='green', alpha=0.15, label='me3')
ax.scatter(xa, ya, s=0.3, c='red',   alpha=0.15, label='ac')
ax.legend(); ax.set_title('me3 vs ac over contour bands'); plt.show()

# (3) Compare core occupancy of the two channels using the true origin
prof_me3 = hmt.simulate.extract_radial_density_profile(me3_df, contour_bands, true_x0, true_y0)
prof_ac  = hmt.simulate.extract_radial_density_profile(ac_df,  contour_bands, true_x0, true_y0)
print("me3 zero-density bands:", np.where(prof_me3==0)[0].min() if (prof_me3==0).any() else None,
      "…count", int((prof_me3==0).sum()))
print("ac  zero-density bands:", np.where(prof_ac==0)[0].min()  if (prof_ac==0).any()  else None,
      "…count", int((prof_ac==0).sum()))


In [ ]:
x_min, y_min = hmt.preprocess.mask_origin(me3_filtered_df, ac_filtered_df)
me3_domains, me3_full, info = hmt.simulate.generate_nucleus(
    me3_df, contour_bands, x_min, y_min, match_spacing=True, rng=np.random.default_rng(0), domain_scale=100)

print("f_noise used:", round(info["f_noise"],3),
      "| from g(r):", round(info["f_noise_from_g"],3),
      "| heuristic:", round(info["f_noise_heuristic"],3))

# apples-to-apples: full simulation (with noise) vs real
hmt.visualize.plot_pair_correlation(info["pcf_real"], info["pcf_noisy"], r_domain=info["r_domain_nm"])


In [ ]:
hmt.visualize.plot_pair_correlation(info["pcf_real"], info["pcf_noisy"], r_domain=info["r_domain_nm"])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
hmt.visualize.plot_nanodomain_2d(me3_domains, info["seeds"], title="Domains only", plot_seeds=False, ax=axes[0])
hmt.visualize.plot_nanodomain_2d(me3_full,    info["seeds"], title="With noise",   plot_seeds=False, ax=axes[1])
plt.tight_layout(); plt.show()

In [ ]:
step = 10
rdf, _ = hmt.simulate.extract_empirical_parameters(me3_df, sdis=500, step=step)
r = (np.arange(len(rdf)) + 0.5) * step

# v1-style width: first radius where the ring density drops below 20% of peak, x2 = diameter
below = np.where(rdf < 0.2 * rdf.max())[0]
width_radius = r[below[0]] if len(below) else r[-1]
print(f"RDF drops <20% of peak at r ≈ {width_radius:.0f} nm  →  domain diameter ≈ {2*width_radius:.0f} nm")

# L(r) peak (Ripley cluster-radius estimator)
x_min, y_min = hmt.preprocess.mask_origin(me3_filtered_df, ac_filtered_df)
pcf = hmt.simulate.measure_pair_correlation(me3_df, contour_bands >= 0, x_min, y_min)
Lr = pcf["L"] - pcf["r"]
print(f"L(r)-r peaks at r ≈ {pcf['r'][np.nanargmax(Lr)]:.0f} nm")

fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].plot(r, rdf); ax[0].axhline(0.2*rdf.max(), ls=':', c='gray'); ax[0].axvline(width_radius, ls=':', c='green')
ax[0].set_xlabel('separation r (nm)'); ax[0].set_ylabel('RDF (locs/nm²)'); ax[0].set_title('me3 RDF — where does it decay?')
ax[1].plot(pcf['r'], Lr); ax[1].axhline(0, ls='--', c='gray')
ax[1].set_xlabel('r (nm)'); ax[1].set_ylabel('L(r)-r'); ax[1].set_title('Ripley L')
plt.tight_layout(); plt.show()


In [ ]:
# Origin the mask grid was built on — pre-mask inputs, NOT the masked me3_df/ac_df
x_min, y_min = hmt.preprocess.mask_origin(me3_filtered_df, ac_filtered_df)

rng = np.random.default_rng(0)
me3_domains, me3_full, info = hmt.simulate.generate_nucleus(
    me3_df, contour_bands, x_min, y_min, match_spacing=True, rng=rng)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
hmt.visualize.plot_nanodomain_2d(me3_domains, info["seeds"], title="Domains only", plot_seeds=False, ax=axes[0])
hmt.visualize.plot_nanodomain_2d(me3_full,    info["seeds"], title="With noise",   plot_seeds=False, ax=axes[1])
plt.tight_layout(); plt.show()

hmt.visualize.plot_pair_correlation(info["pcf_real"], info["pcf_domains"], r_domain=info["r_domain_nm"])


In [ ]:
me3_rdf, me3_adf = hmt.simulate.extract_empirical_parameters(me3_df, sdis=500, step=step)
hmt.visualize.plot_rdf_adf(me3_rdf, me3_adf, ac_rdf, ac_adf, step=10)

In [ ]:
x_min, y_min = hmt.preprocess.mask_origin(me3_filtered_df, ac_filtered_df)

me3_domains, me3_full, info = hmt.simulate.generate_nucleus(
    me3_df, contour_bands, x_min, y_min, match_spacing=True, rng=np.random.default_rng(0), domain_scale=100)
ac_domains, ac_full, info = hmt.simulate.generate_nucleus(
    ac_df, contour_bands, x_min, y_min, match_spacing=True, rng=np.random.default_rng(0), domain_scale=100)

In [ ]:
hmt.simulate.export_to_thunderstorm(me3_domains, me3_df, "simulated_data/me3_domains.csv")
hmt.simulate.export_to_thunderstorm(me3_full, me3_df, "simulated_data/me3_full.csv")

hmt.simulate.export_to_thunderstorm(ac_domains, ac_df, "simulated_data/ac_domains.csv")
hmt.simulate.export_to_thunderstorm(ac_full, ac_df, "simulated_data/ac_full.csv")

In [ ]:
me3_df.to_csv("simulated_data/me3_real.csv", index=False)
ac_df.to_csv("simulated_data/ac_real.csv", index=False)